<a href="https://colab.research.google.com/github/BhumiPatil1507/BTP-/blob/main/TSb_MoE_RidgeStack_Parted_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install packages, import libraries, and mount Google Drive


In [ ]:
# ============================================================
# TSb EXPERT MoE + RIDGESTACK
#
# Workflow:
#   1. Read TSb from the base CSV
#   2. Reproducible 80/20 train-test split
#   3. Train four XGBoost experts
#   4. Generate 3-fold out-of-fold (OOF) predictions on training rows
#   5. Train ExtraTrees MoE router and RidgeStack using OOF predictions
#   6. Evaluate all six models on the untouched test set
#   7. Calculate disagreement uncertainty Di and save outputs
# ============================================================

%pip -q install xgboost scikit-learn scipy joblib

import os
import csv
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib

from google.colab import drive
from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------
drive.mount("/content/drive")

# ------------------------------------------------------------


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configure paths and verify input CSV files


In [ ]:
# 2. CONFIGURATION — CHANGE ONLY IF YOUR FILES ARE ELSEWHERE
# ------------------------------------------------------------
PROJECT_DIR = "/content/drive/MyDrive/Colab"
DATA_FILE = os.path.join(PROJECT_DIR, "polyOne_aa.csv")
FEATURE_DIR = os.path.join(PROJECT_DIR, "polymer_features")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "TSb_MOE_RIDGESTACK_results")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")
OOF_DIR = os.path.join(OUTPUT_DIR, "oof")
SPLIT_DIR = os.path.join(OUTPUT_DIR, "splits")

for folder in [OUTPUT_DIR, MODEL_DIR, PRED_DIR, OOF_DIR, SPLIT_DIR]:
    os.makedirs(folder, exist_ok=True)

TARGET = "TSb"
SEED = 42
TEST_SIZE = 0.20
N_FOLDS = 3
CHUNK_SIZE = 5000

# The validation-based expert training starts with this maximum.
MAX_TREES = 2000
VALIDATION_SIZE = 0.10

EXPERTS = ["fingerprints", "descriptors", "graph", "functional_groups"]

# These names match the Cp notebook provided.
FEATURE_FILES = {
    "fingerprints": os.path.join(FEATURE_DIR, "polyOne_aa_fingerprints.csv"),
    "descriptors": os.path.join(FEATURE_DIR, "polyOne_aa_descriptors.csv"),
    "graph": os.path.join(FEATURE_DIR, "polyOne_aa_graph.csv"),
    "functional_groups": os.path.join(FEATURE_DIR, "polyOne_aa_funcgroups.csv"),
}

# ------------------------------------------------------------
# 3. VERIFY INPUT FILES
# ------------------------------------------------------------
for path in [DATA_FILE, *FEATURE_FILES.values()]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}\n"
            "Check PROJECT_DIR, FEATURE_DIR, and feature CSV filenames."
        )

print("Target:", TARGET)
print("Base CSV:", DATA_FILE)
print("Feature directory:", FEATURE_DIR)
print("Output directory:", OUTPUT_DIR)
print("All required input files found.")

# ------------------------------------------------------------


Target: TSb
Base CSV: /content/drive/MyDrive/Colab/polyOne_aa.csv
Feature directory: /content/drive/MyDrive/Colab/polymer_features
Output directory: /content/drive/MyDrive/Colab/TSb_MOE_RIDGESTACK_results
All required input files found.


## 3. Read TSb and create the reproducible train/test split


In [ ]:
# 4. READ TARGET AND CREATE TRAIN/TEST SPLIT
# ------------------------------------------------------------
target_df = pd.read_csv(DATA_FILE, usecols=[TARGET], low_memory=False)
target_all = pd.to_numeric(target_df[TARGET], errors="coerce").to_numpy(dtype=np.float64)
del target_df
gc.collect()

valid_rows = np.flatnonzero(np.isfinite(target_all))
y = target_all[valid_rows].astype(np.float32)
positions = np.arange(len(y))

if len(y) < 10:
    raise ValueError(f"Only {len(y)} rows have valid {TARGET}; insufficient for training.")

train_idx, test_idx = train_test_split(
    positions,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True
)

y_train = y[train_idx]
y_test = y[test_idx]
train_source_rows = valid_rows[train_idx]
test_source_rows = valid_rows[test_idx]

# Save split positions and original source-row positions for reproducibility.
np.save(os.path.join(SPLIT_DIR, "TSb_valid_source_rows.npy"), valid_rows)
np.save(os.path.join(SPLIT_DIR, "TSb_train_indices.npy"), train_idx)
np.save(os.path.join(SPLIT_DIR, "TSb_test_indices.npy"), test_idx)
np.save(os.path.join(SPLIT_DIR, "TSb_train_source_rows.npy"), train_source_rows)
np.save(os.path.join(SPLIT_DIR, "TSb_test_source_rows.npy"), test_source_rows)

print(f"\nRows with valid TSb: {len(y):,}")
print(f"Train rows: {len(train_idx):,} ({len(train_idx)/len(y):.1%})")
print(f"Test rows:  {len(test_idx):,} ({len(test_idx)/len(y):.1%})")
print(f"TSb range: {float(np.min(y)):.6g} to {float(np.max(y)):.6g}")

# ------------------------------------------------------------



Rows with valid TSb: 500,000
Train rows: 400,000 (80.0%)
Test rows:  100,000 (20.0%)
TSb range: 9.27672 to 173.442


## 4. Identify extracted features and define the feature loader


In [ ]:
# 5. IDENTIFY EXTRACTED FEATURE COLUMNS
#    Feature CSVs must be row-aligned with the base CSV and contain TSb.
#    Columns already present in the base CSV are not treated as features.
# ------------------------------------------------------------
with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    base_columns = next(csv.reader(f))
base_set = set(base_columns)

FEATURE_COLUMNS = {}
for expert, path in FEATURE_FILES.items():
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        header = next(csv.reader(f))

    if TARGET not in header:
        raise ValueError(
            f"'{TARGET}' is missing from {path}. "
            "The feature CSV must be aligned with the base data and include the target column."
        )

    cols = [col for col in header if col not in base_set]
    if not cols:
        raise ValueError(
            f"No extracted feature columns found for '{expert}'. "
            "Check whether FEATURE_DIR points to the feature-extraction outputs."
        )

    FEATURE_COLUMNS[expert] = cols
    print(f"{expert:20s}: {len(cols):,} extracted features")

with open(os.path.join(OUTPUT_DIR, "TSb_feature_columns.json"), "w") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

# ------------------------------------------------------------
# 6. MEMORY-AWARE FEATURE ROW LOADER
#    Loads only the requested original CSV rows and selected feature columns.
# ------------------------------------------------------------
def load_feature_rows(expert, requested_source_rows):
    path = FEATURE_FILES[expert]
    cols = FEATURE_COLUMNS[expert]
    requested_source_rows = np.asarray(requested_source_rows, dtype=np.int64)

    if len(requested_source_rows) == 0:
        return np.empty((0, len(cols)), dtype=np.float32)

    # Read requested rows in ascending order, then restore requested order.
    order = np.argsort(requested_source_rows)
    sorted_rows = requested_source_rows[order]
    X_sorted = np.empty((len(sorted_rows), len(cols)), dtype=np.float32)

    left = 0
    csv_start = 0

    for chunk in pd.read_csv(
        path, usecols=cols, chunksize=CHUNK_SIZE, low_memory=False
    ):
        csv_end = csv_start + len(chunk)
        right = np.searchsorted(sorted_rows, csv_end, side="left")

        if right > left:
            local_rows = sorted_rows[left:right] - csv_start
            X_sorted[left:right] = chunk.iloc[local_rows].to_numpy(
                dtype=np.float32, copy=True
            )

        left = right
        csv_start = csv_end
        del chunk

        if left >= len(sorted_rows):
            break

    if left != len(sorted_rows):
        raise ValueError(
            f"Could not retrieve all requested rows from {path}. "
            "Check that feature CSV row counts/order match the base CSV."
        )

    inverse = np.empty_like(order)
    inverse[order] = np.arange(len(order))
    X = X_sorted[inverse]

    del X_sorted
    gc.collect()

    if expert == "fingerprints" and not np.isfinite(X).all():
        raise ValueError("Fingerprint feature matrix contains missing/non-finite values.")

    return X

# ------------------------------------------------------------


fingerprints        : 2,048 extracted features
descriptors         : 16 extracted features
graph               : 17 extracted features
functional_groups   : 33 extracted features


## 5. Train the four individual XGBoost experts


In [ ]:
# 7. TRAIN FOUR FINAL EXPERTS
#    A 10% validation subset chooses the number of boosting rounds.
#    Then the expert is refit on the full training split using that
#    selected number of rounds. The test set is not used for fitting.
# ------------------------------------------------------------
final_test_predictions = []
expert_metrics = []
tree_counts = {}

for expert in EXPERTS:
    print("\n" + "=" * 72)
    print("TRAINING FINAL EXPERT:", expert, flush=True)
    started = time.time()

    model_path = os.path.join(MODEL_DIR, f"TSb_{expert}_expert.json")
    pred_path = os.path.join(PRED_DIR, f"TSb_{expert}_test_predictions.npy")
    metrics_path = os.path.join(MODEL_DIR, f"TSb_{expert}_metrics.json")
    imputer_path = os.path.join(MODEL_DIR, f"TSb_{expert}_imputer.joblib")

    X_train_raw = load_feature_rows(expert, train_source_rows)
    X_test_raw = load_feature_rows(expert, test_source_rows)

    imputer = None
    if expert != "fingerprints":
        # Fit imputation on the training split only.
        imputer = SimpleImputer(strategy="median")
        X_train = imputer.fit_transform(X_train_raw).astype(np.float32, copy=False)
        X_test = imputer.transform(X_test_raw).astype(np.float32, copy=False)
        joblib.dump(imputer, imputer_path)
    else:
        X_train = X_train_raw
        X_test = X_test_raw

    del X_train_raw, X_test_raw
    gc.collect()

    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise ValueError(f"Non-finite feature values remain for expert '{expert}'.")

    inner_train, val_pos = train_test_split(
        np.arange(len(y_train)),
        test_size=VALIDATION_SIZE,
        random_state=SEED,
        shuffle=True
    )

    selector = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=MAX_TREES,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="rmse",
        early_stopping_rounds=100,
        n_jobs=-1,
        random_state=SEED,
        verbosity=0
    )

    selector.fit(
        X_train[inner_train],
        y_train[inner_train],
        eval_set=[(X_train[val_pos], y_train[val_pos])],
        verbose=100
    )

    selected_rounds = int(getattr(selector, "best_iteration", MAX_TREES - 1)) + 1
    hit_tree_limit = selected_rounds >= MAX_TREES

    # Refit using all training rows at the validation-selected number of rounds.
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=selected_rounds,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
        verbosity=0
    )
    model.fit(X_train, y_train, verbose=False)

    pred = model.predict(X_test).astype(np.float64)
    metrics = {
        "target": TARGET,
        "expert": expert,
        "feature_count": len(FEATURE_COLUMNS[expert]),
        "R2": float(r2_score(y_test, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
        "MAE": float(mean_absolute_error(y_test, pred)),
        "selected_trees": int(selected_rounds),
        "hit_max_tree_limit": bool(hit_tree_limit),
        "training_minutes": float((time.time() - started) / 60.0)
    }

    model.save_model(model_path)
    np.save(pred_path, pred)
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    final_test_predictions.append(pred)
    expert_metrics.append(metrics)
    tree_counts[expert] = selected_rounds

    print(json.dumps(metrics, indent=2), flush=True)
    if hit_tree_limit:
        print(
            f"WARNING: {expert} selected the MAX_TREES limit ({MAX_TREES}). "
            "Consider increasing MAX_TREES and rerunning if needed.",
            flush=True
        )

    del X_train, X_test, selector, model, imputer
    gc.collect()

test_pred_matrix = np.column_stack(final_test_predictions).astype(np.float64)
np.save(
    os.path.join(PRED_DIR, "TSb_all_expert_test_predictions.npy"),
    test_pred_matrix
)
pd.DataFrame(expert_metrics).to_csv(
    os.path.join(OUTPUT_DIR, "TSb_individual_expert_metrics.csv"),
    index=False
)

print("\nSelected tree counts for OOF models:", tree_counts)

# ------------------------------------------------------------



TRAINING FINAL EXPERT: fingerprints
[0]	validation_0-rmse:16.79583
[100]	validation_0-rmse:9.40183
[200]	validation_0-rmse:8.70831
[300]	validation_0-rmse:8.38694
[400]	validation_0-rmse:8.13847
[500]	validation_0-rmse:7.95002
[600]	validation_0-rmse:7.80552
[700]	validation_0-rmse:7.68099
[800]	validation_0-rmse:7.57829
[900]	validation_0-rmse:7.47606
[1000]	validation_0-rmse:7.39034
[1100]	validation_0-rmse:7.30620
[1200]	validation_0-rmse:7.23820
[1300]	validation_0-rmse:7.17400
[1400]	validation_0-rmse:7.11893
[1500]	validation_0-rmse:7.06926
[1600]	validation_0-rmse:7.02278
[1700]	validation_0-rmse:6.97871
[1800]	validation_0-rmse:6.93590
[1900]	validation_0-rmse:6.89409
[1999]	validation_0-rmse:6.86153
{
  "target": "TSb",
  "expert": "fingerprints",
  "feature_count": 2048,
  "R2": 0.8416422665378274,
  "RMSE": 6.870924131204002,
  "MAE": 5.115645125963092,
  "selected_trees": 2000,
  "hit_max_tree_limit": true,
  "training_minutes": 67.76729383865992
}

TRAINING FINAL EXPERT: 

## 6. Generate out-of-fold (OOF) predictions for stacking


In [ ]:
# 8. GENERATE OOF PREDICTIONS FOR STACKING
#    Each training row is predicted by an expert that did not train on it.
#    Test rows are not involved in creating OOF predictions.
# ------------------------------------------------------------
OOF_PATH = os.path.join(OOF_DIR, "TSb_oof_predictions.npy")
OOF_PROGRESS = os.path.join(OOF_DIR, "TSb_oof_progress.json")

n_train = len(y_train)
oof = np.lib.format.open_memmap(
    OOF_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(n_train, len(EXPERTS))
)
oof[:] = np.nan
oof.flush()

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
completed = []

for fold, (fit_pos, hold_pos) in enumerate(
    kf.split(np.arange(n_train)), start=1
):
    print(
        f"\nOOF fold {fold}/{N_FOLDS}: "
        f"fit={len(fit_pos):,}, holdout={len(hold_pos):,}",
        flush=True
    )

    fit_source = train_source_rows[fit_pos]
    hold_source = train_source_rows[hold_pos]
    y_fit = y_train[fit_pos]

    for j, expert in enumerate(EXPERTS):
        print(f"  OOF expert: {expert}", flush=True)

        X_fit_raw = load_feature_rows(expert, fit_source)
        X_hold_raw = load_feature_rows(expert, hold_source)

        imp = None
        if expert != "fingerprints":
            # Fit a fresh imputer inside this fold to avoid fold leakage.
            imp = SimpleImputer(strategy="median")
            X_fit = imp.fit_transform(X_fit_raw).astype(np.float32, copy=False)
            X_hold = imp.transform(X_hold_raw).astype(np.float32, copy=False)
        else:
            X_fit = X_fit_raw
            X_hold = X_hold_raw

        del X_fit_raw, X_hold_raw
        gc.collect()

        if not np.isfinite(X_fit).all() or not np.isfinite(X_hold).all():
            raise ValueError(f"Non-finite values found in OOF features for {expert}.")

        model = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=tree_counts[expert],
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED + fold,
            verbosity=0
        )

        model.fit(X_fit, y_fit, verbose=False)
        oof[hold_pos, j] = model.predict(X_hold).astype(np.float32)
        oof.flush()

        completed.append(f"{expert}_fold_{fold}")
        with open(OOF_PROGRESS, "w") as f:
            json.dump(
                {
                    "target": TARGET,
                    "n_folds": N_FOLDS,
                    "completed_tasks": completed,
                    "completed_count": len(completed),
                    "total_tasks": N_FOLDS * len(EXPERTS)
                },
                f,
                indent=2
            )

        del X_fit, X_hold, model, imp
        gc.collect()

    del fit_source, hold_source, y_fit
    gc.collect()

oof.flush()

if not np.isfinite(oof).all():
    raise ValueError("OOF predictions contain non-finite values. Do not train stackers.")

print(
    f"\nOOF complete: shape={oof.shape}; "
    f"tasks={len(completed)}/{N_FOLDS * len(EXPERTS)}",
    flush=True
)
del oof
gc.collect()

# ------------------------------------------------------------



OOF fold 1/3: fit=266,666, holdout=133,334
  OOF expert: fingerprints
  OOF expert: descriptors
  OOF expert: graph
  OOF expert: functional_groups

OOF fold 2/3: fit=266,667, holdout=133,333
  OOF expert: fingerprints
  OOF expert: descriptors
  OOF expert: graph
  OOF expert: functional_groups

OOF fold 3/3: fit=266,667, holdout=133,333
  OOF expert: fingerprints
  OOF expert: descriptors
  OOF expert: graph
  OOF expert: functional_groups

OOF complete: shape=(400000, 4); tasks=12/12


0

## 7. Train the ExtraTrees MoE router and RidgeStack


In [ ]:
# 9. TRAIN MoE ROUTER AND RIDGESTACK USING OOF PREDICTIONS
# ------------------------------------------------------------
oof_matrix = np.load(OOF_PATH, mmap_mode="r")

if oof_matrix.shape != (len(y_train), len(EXPERTS)):
    raise ValueError(f"Unexpected OOF shape: {oof_matrix.shape}")
if not np.isfinite(oof_matrix).all():
    raise ValueError("OOF prediction matrix contains non-finite values.")

# Nonlinear MoE-style router over the four expert predictions.
router = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    max_features=1.0,
    n_jobs=-1,
    random_state=SEED
)
router.fit(np.asarray(oof_matrix), y_train)
joblib.dump(router, os.path.join(MODEL_DIR, "TSb_MoE_router.joblib"))
moe_pred = router.predict(test_pred_matrix)

# RidgeStack: RidgeCV selects alpha using training OOF predictions only.
ridge_stack = RidgeCV(alphas=np.logspace(-4, 4, 25), cv=5)
ridge_stack.fit(np.asarray(oof_matrix), y_train)
joblib.dump(ridge_stack, os.path.join(MODEL_DIR, "TSb_RidgeStack.joblib"))
ridge_pred = ridge_stack.predict(test_pred_matrix)

# ------------------------------------------------------------


## 8. Evaluate models, calculate disagreement Di, and save results


In [ ]:
# 10. TEST METRICS FOR ALL FOUR EXPERTS + BOTH STACKERS
# ------------------------------------------------------------
def metrics_row(name, truth, pred):
    return {
        "Model": name,
        "R2": float(r2_score(truth, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(truth, pred))),
        "MAE": float(mean_absolute_error(truth, pred))
    }

rows = [
    metrics_row(expert, y_test, test_pred_matrix[:, i])
    for i, expert in enumerate(EXPERTS)
]
rows.append(metrics_row("MoE_ExtraTrees", y_test, moe_pred))
rows.append(metrics_row("RidgeStack", y_test, ridge_pred))

metrics_df = pd.DataFrame(rows)
metrics_path = os.path.join(OUTPUT_DIR, "TSb_experts_MoE_RidgeStack_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

# ------------------------------------------------------------
# 11. DISAGREEMENT UNCERTAINTY (Di) AND ERROR CORRELATIONS
#    Di = sample standard deviation across four expert predictions.
#    Di is a disagreement score, not a calibrated prediction interval.
# ------------------------------------------------------------
Di = np.std(test_pred_matrix, axis=1, ddof=1)
moe_abs_error = np.abs(y_test - moe_pred)
ridge_abs_error = np.abs(y_test - ridge_pred)

def safe_corr(a, b):
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

uncertainty_df = pd.DataFrame({
    "Source_Row": test_source_rows,
    "Actual_TSb": y_test,
    "MoE_Predicted_TSb": moe_pred,
    "MoE_Absolute_Error": moe_abs_error,
    "RidgeStack_Predicted_TSb": ridge_pred,
    "RidgeStack_Absolute_Error": ridge_abs_error,
    "Disagreement_Di": Di
})

for i, expert in enumerate(EXPERTS):
    uncertainty_df[f"{expert}_Prediction"] = test_pred_matrix[:, i]

uncertainty_path = os.path.join(
    PRED_DIR, "TSb_test_predictions_Di_and_errors.csv"
)
uncertainty_df.to_csv(uncertainty_path, index=False)

summary = {
    "target": TARGET,
    "n_valid": int(len(y)),
    "n_train": int(len(y_train)),
    "n_test": int(len(y_test)),
    "split_seed": SEED,
    "test_size": TEST_SIZE,
    "n_oof_folds": N_FOLDS,
    "selected_tree_counts": tree_counts,
    "ridge_alpha_selected": float(ridge_stack.alpha_),
    "Di_mean": float(np.mean(Di)),
    "Di_median": float(np.median(Di)),
    "Di_p95": float(np.percentile(Di, 95)),
    "corr_Di_MoE_absolute_error": safe_corr(Di, moe_abs_error),
    "corr_Di_RidgeStack_absolute_error": safe_corr(Di, ridge_abs_error)
}

with open(os.path.join(OUTPUT_DIR, "TSb_run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# ------------------------------------------------------------
# 12. REPORT RESULTS AND OUTPUT LOCATIONS
# ------------------------------------------------------------
print("\n" + "=" * 88)
print("TSb: INDIVIDUAL EXPERTS vs MoE vs RIDGESTACK — HELD-OUT TEST SET")
print("=" * 88)
print(metrics_df.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("\nRidgeStack selected alpha:", ridge_stack.alpha_)
print("\nDISAGREEMENT UNCERTAINTY")
print("Di = sample standard deviation across the four individual expert predictions")
print(f"Mean Di: {summary['Di_mean']:.6f}")
print(f"Median Di: {summary['Di_median']:.6f}")
print(f"95th percentile Di: {summary['Di_p95']:.6f}")
print(
    "Correlation Di vs MoE absolute error: "
    f"{summary['corr_Di_MoE_absolute_error']:.6f}"
)
print(
    "Correlation Di vs RidgeStack absolute error: "
    f"{summary['corr_Di_RidgeStack_absolute_error']:.6f}"
)

print("\nSaved outputs:")
print("Metrics CSV:       ", metrics_path)
print("Predictions + Di:  ", uncertainty_path)
print("Models directory:  ", MODEL_DIR)
print("OOF predictions:   ", OOF_PATH)
print("Split indices:     ", SPLIT_DIR)
print("Run summary:       ", os.path.join(OUTPUT_DIR, "TSb_run_summary.json"))
print("\nDONE.")



TSb: INDIVIDUAL EXPERTS vs MoE vs RIDGESTACK — HELD-OUT TEST SET
            Model       R2      RMSE       MAE
     fingerprints 0.841642  6.870924  5.115645
      descriptors 0.691576  9.588924  7.290475
            graph 0.596974 10.961301  8.376623
functional_groups 0.376433 13.634432 10.309833
   MoE_ExtraTrees 0.852068  6.640885  4.971027
       RidgeStack 0.850054  6.685956  5.001808

RidgeStack selected alpha: 10.0

DISAGREEMENT UNCERTAINTY
Di = sample standard deviation across the four individual expert predictions
Mean Di: 5.707923
Median Di: 5.006536
95th percentile Di: 12.059233
Correlation Di vs MoE absolute error: 0.158539
Correlation Di vs RidgeStack absolute error: 0.172076

Saved outputs:
Metrics CSV:        /content/drive/MyDrive/Colab/TSb_MOE_RIDGESTACK_results/TSb_experts_MoE_RidgeStack_metrics.csv
Predictions + Di:   /content/drive/MyDrive/Colab/TSb_MOE_RIDGESTACK_results/predictions/TSb_test_predictions_Di_and_errors.csv
Models directory:   /content/drive/MyDrive